# 03 · Grouped-query attention (GQA) and multi-query attention (MQA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/03_gqa_mqa.ipynb) &nbsp; 📖 Concept deck: [KV cache](https://nilesh-salpe.github.io/visual-ai/inference/02-kv-cache/) &nbsp; ⏱️ Suggested time: **25 min**

**You will build:** attention where several query heads share each key/value head (GQA, Ainslie et al. 2023). Multi-head attention and MQA are the two extremes.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def _ref_gqa(x, Wq, Wk, Wv, Wo, H, Hkv):
    T, d = x.shape; dh = Wq.shape[1] // H
    q = (x @ Wq).reshape(T, H, dh); k = (x @ Wk).reshape(T, Hkv, dh); v = (x @ Wv).reshape(T, Hkv, dh)
    mask = np.tril(np.ones((T, T), bool)); out = np.zeros((T, H, dh))
    for h in range(H):
        g = h // (H // Hkv)                       # the kv head this query head reads
        s = q[:, h] @ k[:, g].T / np.sqrt(dh)
        out[:, h] = softmax(np.where(mask, s, -np.inf)) @ v[:, g]
    return out.reshape(T, H * dh) @ Wo

def make_weights(d=32, H=8, Hkv=2, seed=0):
    r = np.random.default_rng(seed); dh = d // H
    return r.normal(0, .3, (d, H * dh)), r.normal(0, .3, (d, Hkv * dh)), r.normal(0, .3, (d, Hkv * dh)), r.normal(0, .3, (H * dh, d))
print('helpers ready: make_weights(d, H, Hkv)')

## Task 1: GQA forward pass

`Wq` projects to `H` query heads; `Wk` and `Wv` project to only `Hkv` key/value heads (`H % Hkv == 0`).
Query head `h` uses kv head `h // (H // Hkv)`. Apply a causal mask, then merge the heads and apply `@ Wo`.
Ideally avoid a Python loop over heads: `np.repeat` the kv heads along the head axis.

In [ ]:
def gqa_attention(x, Wq, Wk, Wv, Wo, n_heads, n_kv_heads):
    """x: [T, d] -> [T, d]. Causal."""
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_gqa_attention(f):
    x = np.random.default_rng(3).normal(size=(6, 32))
    for H, Hkv in [(8, 8), (8, 2), (8, 1)]:
        W = make_weights(32, H, Hkv, seed=H + Hkv)
        assert np.allclose(f(x, *W, H, Hkv), _ref_gqa(x, *W, H, Hkv), atol=1e-6), f"mismatch for H={H}, Hkv={Hkv} (query head h must use kv head h // (H // Hkv))"

check("gqa_attention", _test_gqa_attention, gqa_attention)

## Task 2: How much smaller is the cache?

Return the factor by which GQA shrinks the KV cache compared with multi-head attention that has the same number of query heads.

In [ ]:
def kv_cache_reduction(n_heads, n_kv_heads):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_kv_cache_reduction(f):
    assert f(64, 8) == 8, "64 query heads sharing 8 kv heads -> cache is 8x smaller"
    assert f(32, 32) == 1 and f(32, 1) == 32, "MHA -> 1x, MQA -> n_heads x"

check("kv_cache_reduction", _test_kv_cache_reduction, kv_cache_reduction)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: GQA forward pass
def gqa_attention_solution(x, Wq, Wk, Wv, Wo, n_heads, n_kv_heads):
    T, d = x.shape; dh = Wq.shape[1] // n_heads; rep = n_heads // n_kv_heads
    q = (x @ Wq).reshape(T, n_heads, dh).transpose(1, 0, 2)                      # [H, T, dh]
    k = (x @ Wk).reshape(T, n_kv_heads, dh).transpose(1, 0, 2)                   # [Hkv, T, dh]
    v = (x @ Wv).reshape(T, n_kv_heads, dh).transpose(1, 0, 2)
    k, v = np.repeat(k, rep, axis=0), np.repeat(v, rep, axis=0)                 # share each kv head with `rep` query heads
    s = q @ k.transpose(0, 2, 1) / np.sqrt(dh)
    s = np.where(np.tril(np.ones((T, T), bool)), s, -np.inf)
    return (softmax(s) @ v).transpose(1, 0, 2).reshape(T, n_heads * dh) @ Wo

check("gqa_attention (reference)", _test_gqa_attention, gqa_attention_solution)

In [ ]:
# @title Solution for Task 2: How much smaller is the cache?
def kv_cache_reduction_solution(n_heads, n_kv_heads):
    return n_heads // n_kv_heads

check("kv_cache_reduction (reference)", _test_kv_cache_reduction, kv_cache_reduction_solution)

---
## Takeaways
- The KV cache scales with the number of **kv heads**, not query heads. GQA with 8 kv heads for 64 query heads stores 8× less.
- MHA (`Hkv = H`) and MQA (`Hkv = 1`) are the two ends of one family. GQA can be "uptrained" from an MHA checkpoint with about 5% of the original pre-training compute (Ainslie et al. 2023).